
# Practical Encoding Techniques on Telco Customer Churn Dataset

## Objective
This notebook demonstrates multiple encoding techniques used in machine learning preprocessing.

We will:
- Load and explore the Telco Customer Churn dataset
- Perform basic preprocessing
- Split the data into train and test sets
- Apply different encoding methods
- Compare how each encoding affects the dataset

Encoding techniques covered:
- Label Encoding
- One Hot Encoding
- Frequency Encoding
- Target Encoding


## 1. Import Required Libraries

In [ ]:

import pandas as pd
import  seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

## 2. Load Dataset

In [ ]:

df = pd.read_csv("Telco-Customer-Churn.csv")
df.head()


## 3. Dataset Inspection

In [ ]:
df.shape

In [ ]:
df.info()


## 4. Basic Preprocessing
Remove identifier column and fix datatype issues.


In [ ]:

df.drop("customerID", axis=1, inplace=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


## 5. Separate Features and Target

In [ ]:

X = df.drop("Churn", axis=1)
y = df["Churn"].map({"Yes":1, "No":0})



## 6. Train Test Split
We split before encoding to avoid data leakage.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



## 7. Label Encoding (Binary Features)
Used for features with only two categories.


In [ ]:

binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling']

X_train_label = X_train.copy()
X_test_label = X_test.copy()

le = LabelEncoder()

for col in binary_cols:
    X_train_label[col] = le.fit_transform(X_train_label[col])
    X_test_label[col] = le.transform(X_test_label[col])



## 8. One Hot Encoding
Suitable for nominal categorical features without inherent order.


In [ ]:

nominal_cols = ['InternetService','Contract','PaymentMethod','MultipleLines','gender']

X_train_ohe = pd.get_dummies(X_train, columns=nominal_cols, drop_first=True)
X_test_ohe = pd.get_dummies(X_test, columns=nominal_cols, drop_first=True)



## 9. Frequency Encoding
Replaces categories with their occurrence counts.


In [ ]:

X_train_freq = X_train.copy()
X_test_freq = X_test.copy()

freq_map = X_train_freq['PaymentMethod'].value_counts()

X_train_freq['PaymentMethod'] = X_train_freq['PaymentMethod'].map(freq_map)
X_test_freq['PaymentMethod'] = X_test_freq['PaymentMethod'].map(freq_map)



## 10. Target Encoding
Replaces a category with the mean value of the target variable for that category.


In [ ]:

X_train_target = X_train.copy()
X_test_target = X_test.copy()

target_mean = pd.concat([X_train, y_train], axis=1).groupby('InternetService')['Churn'].mean()

X_train_target['InternetService'] = X_train_target['InternetService'].map(target_mean)
X_test_target['InternetService'] = X_test_target['InternetService'].map(target_mean)



## 12. Encoding Comparison

| Encoding | Dimensionality | Use Case |
|--------|--------|--------|
| Label Encoding | Low | Binary categories |
| One Hot Encoding | High | Nominal categories |
| Frequency Encoding | Low | High cardinality |
| Target Encoding | Low | Captures relationship with target |



## 13. Key Insights

- One Hot Encoding increases the number of features significantly.
- Label Encoding works best for binary categorical variables.
- Frequency Encoding avoids dimensionality explosion.
- Target Encoding can capture relationships with the target but must be used carefully to avoid data leakage.


## 14. Model Executionn

If we try to implement a model on this dataset so we need to merge the Encoding columns of `X_train_ohe` and  `X_train_label` because they have contain different type of encoding colummns, if we only fit anyone of these data so it gaves error.